In [1]:
%pip install optuna

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score,
                             precision_score, recall_score, f1_score)
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

print("All libraries imported successfully")

All libraries imported successfully


In [3]:
train = pd.read_csv('/Users/deannalakshman/Desktop/IIT/YEAR 2/SEM 2/DSPLC/Coursework technical/EDA/featured_dataset/train_featured.csv')
val   = pd.read_csv('/Users/deannalakshman/Desktop/IIT/YEAR 2/SEM 2/DSPLC/Coursework technical/EDA/featured_dataset/val_featured.csv')


print(f"Train      : {train.shape}")
print(f"Validation : {val.shape}")

Train      : (27499, 37)
Validation : (2749, 37)


In [4]:
for df in [train, val]:
    df['Target'] = np.where(df['Reservation_Status'] == 'Cancelled', 1,
                   np.where(df['Reservation_Status'] == 'No-Show', 2, 0))

labels = {0: 'Not Cancelled', 1: 'Cancelled', 2: 'No-Show'}

print("=== Class Distribution — Train ===")
for cls, count in train['Target'].value_counts().sort_index().items():
    print(f"  {cls} ({labels[cls]:<15}) : {count}  ({count/len(train)*100:.1f}%)")

print("\n=== Class Distribution — Validation ===")
for cls, count in val['Target'].value_counts().sort_index().items():
    print(f"  {cls} ({labels[cls]:<15}) : {count}  ({count/len(val)*100:.1f}%)")

=== Class Distribution — Train ===
  0 (Not Cancelled  ) : 21240  (77.2%)
  1 (Cancelled      ) : 4134  (15.0%)
  2 (No-Show        ) : 2125  (7.7%)

=== Class Distribution — Validation ===
  0 (Not Cancelled  ) : 1610  (58.6%)
  1 (Cancelled      ) : 741  (27.0%)
  2 (No-Show        ) : 398  (14.5%)


In [5]:
date_cols = ['Expected_checkin', 'Expected_checkout', 'Booking_date']

for df in [train, val]:
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    df['Lead_Time']         = (df['Expected_checkin'] - df['Booking_date']).dt.days
    df['Length_of_Stay']    = (df['Expected_checkout'] - df['Expected_checkin']).dt.days
    df['Length_of_Stay']    = df['Length_of_Stay'].clip(lower=1)
    df['Booking_Month']     = df['Booking_date'].dt.month
    df['Checkin_Month']     = df['Expected_checkin'].dt.month
    df['Checkin_DayOfWeek'] = df['Expected_checkin'].dt.dayofweek

# Drop date and status columns
drop_cols = ['Reservation_Status', 'Expected_checkin', 'Expected_checkout', 'Booking_date']
train = train.drop(columns=drop_cols, errors='ignore')
val   = val.drop(columns=drop_cols, errors='ignore')

print("Date features extracted")
print(f"Train shape : {train.shape}")

Date features extracted
Train shape : (27499, 35)


In [6]:
median_rate = train['Room_Rate'].median()

for df in [train, val]:
    df['Total_Guests']         = df['Adults'] + df['Children'] + df['Babies']
    df['Has_Discount']         = (df['Discount_Rate'] > 0).astype(int)
    df['High_Room_Rate']       = (df['Room_Rate'] > median_rate).astype(int)
    df['Long_Lead_Time']       = (df['Lead_Time'] > 90).astype(int)
    df['Short_Stay']           = (df['Length_of_Stay'] == 1).astype(int)
    df['Weekend_Checkin']      = (df['Checkin_DayOfWeek'] >= 5).astype(int)
    df['Peak_Season']          = df['Checkin_Month'].isin([6, 7, 8]).astype(int)
    df['Has_Cancelled_Before'] = (df['Previous_Cancellations'] == 'Yes').astype(int)

print("Feature engineering complete")

Feature engineering complete


In [7]:
cat_cols = [c for c in train.select_dtypes(include='object').columns
            if c not in ['Target']]
print("Encoding:", cat_cols)

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], val[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    val[col]   = le.transform(val[col].astype(str))

print("Encoding complete")

Encoding: ['Gender', 'Ethnicity', 'Educational_Level', 'Income', 'Country_region', 'Hotel_Type', 'Meal_Type', 'Visted_Previously', 'Previous_Cancellations', 'Deposit_type', 'Booking_channel', 'Required_Car_Parking', 'Use_Promotion']
Encoding complete


In [8]:
drop_cols = ['Target', 'Reservation-id', 'Reservation_Status_Code',
             'Cancelled', 'No_Show']

X_train = train.drop(columns=[c for c in drop_cols if c in train.columns])
y_train = train['Target']

X_val = val.drop(columns=[c for c in drop_cols if c in val.columns])
y_val = val['Target']

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"Features: {X_train.columns.tolist()}")

X_train : (27499, 37)
X_val   : (2749, 37)
Features: ['Gender', 'Age', 'Ethnicity', 'Educational_Level', 'Income', 'Country_region', 'Hotel_Type', 'Adults', 'Children', 'Babies', 'Meal_Type', 'Visted_Previously', 'Previous_Cancellations', 'Deposit_type', 'Booking_channel', 'Required_Car_Parking', 'Use_Promotion', 'Discount_Rate', 'Room_Rate', 'Lead_Time', 'Stay_Duration', 'Booking_Month', 'Checkin_Month', 'Checkin_DayOfWeek', 'Is_Weekend_Checkin', 'Total_Guests', 'Has_Children', 'Has_Babies', 'Net_Room_Rate', 'Length_of_Stay', 'Has_Discount', 'High_Room_Rate', 'Long_Lead_Time', 'Short_Stay', 'Weekend_Checkin', 'Peak_Season', 'Has_Cancelled_Before']


In [9]:
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

print("Scaling complete — fitted on train only")

Scaling complete — fitted on train only


In [10]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

print("Class distribution BEFORE SMOTE:")
for cls, count in pd.Series(y_train).value_counts().sort_index().items():
    print(f"  {labels[cls]:<15} : {count}")

print("\nClass distribution AFTER SMOTE:")
for cls, count in pd.Series(y_train_sm).value_counts().sort_index().items():
    print(f"  {labels[cls]:<15} : {count}")

Class distribution BEFORE SMOTE:
  Not Cancelled   : 21240
  Cancelled       : 4134
  No-Show         : 2125

Class distribution AFTER SMOTE:
  Not Cancelled   : 21240
  Cancelled       : 21240
  No-Show         : 21240


In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

CLASS_NAMES = ['Not Cancelled', 'Cancelled', 'No-Show']

def evaluate_model(name, model, X, y, color='Blues'):
    preds = model.predict(X)
    probs = model.predict_proba(X)

    acc  = accuracy_score(y, preds)
    prec = precision_score(y, preds, average='weighted', zero_division=0)
    rec  = recall_score(y, preds, average='weighted', zero_division=0)
    f1   = f1_score(y, preds, average='weighted', zero_division=0)
    try:
        auc = roc_auc_score(y, probs, multi_class='ovr', average='weighted')
    except:
        auc = float('nan')

    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}  <- primary metric")
    print(f"  F1 Score  : {f1:.4f}  <- secondary metric")
    print(f"  ROC AUC   : {auc:.4f}")
    print()
    print(classification_report(y, preds, target_names=CLASS_NAMES))

    cm = confusion_matrix(y, preds)
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap=color, ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(f"{name} — Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    plt.tight_layout()
    plt.show()

    return {'Accuracy': round(acc,4), 'Precision': round(prec,4),
            'Recall': round(rec,4), 'F1 Score': round(f1,4),
            'ROC AUC': round(auc,4)}

print("Evaluation function ready")

Evaluation function ready


In [12]:
# ============================================================
# OPTUNA OBJECTIVE FUNCTIONS
# Optuna searches smarter than GridSearch — it learns from
# previous trials and focuses on promising parameter regions
# n_trials=50 means it tries 50 combinations per model
# ============================================================

def objective_lr(trial):
    C      = trial.suggest_float('C', 0.001, 100, log=True)
    solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear'])
    model  = LogisticRegression(C=C, solver=solver, class_weight='balanced',
                                 max_iter=1000, random_state=42)
    score  = cross_val_score(model, X_train_sm, y_train_sm,
                              cv=cv, scoring='f1_weighted', n_jobs=-1).mean()
    return score

def objective_rf(trial):
    n_estimators     = trial.suggest_int('n_estimators', 100, 500, step=100)
    max_depth        = trial.suggest_int('max_depth', 5, 30)
    min_samples_split= trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 5)
    model  = RandomForestClassifier(n_estimators=n_estimators,
                                     max_depth=max_depth,
                                     min_samples_split=min_samples_split,
                                     min_samples_leaf=min_samples_leaf,
                                     class_weight='balanced',
                                     random_state=42, n_jobs=-1)
    score  = cross_val_score(model, X_train_sm, y_train_sm,
                              cv=cv, scoring='f1_weighted', n_jobs=-1).mean()
    return score

def objective_xgb(trial):
    n_estimators  = trial.suggest_int('n_estimators', 100, 500, step=100)
    max_depth     = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    subsample     = trial.suggest_float('subsample', 0.6, 1.0)
    colsample     = trial.suggest_float('colsample_bytree', 0.6, 1.0)
    model  = XGBClassifier(n_estimators=n_estimators,
                            max_depth=max_depth,
                            learning_rate=learning_rate,
                            subsample=subsample,
                            colsample_bytree=colsample,
                            objective='multi:softprob',
                            num_class=3,
                            eval_metric='mlogloss',
                            random_state=42, verbosity=0)
    score  = cross_val_score(model, X_train_sm, y_train_sm,
                              cv=cv, scoring='f1_weighted', n_jobs=-1).mean()
    return score

def objective_dt(trial):
    max_depth         = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 5)
    criterion         = trial.suggest_categorical('criterion', ['gini', 'entropy'])
    model  = DecisionTreeClassifier(max_depth=max_depth,
                                     min_samples_split=min_samples_split,
                                     min_samples_leaf=min_samples_leaf,
                                     criterion=criterion,
                                     class_weight='balanced',
                                     random_state=42)
    score  = cross_val_score(model, X_train_sm, y_train_sm,
                              cv=cv, scoring='f1_weighted', n_jobs=-1).mean()
    return score

print("Optuna objective functions defined")

Optuna objective functions defined


In [13]:
# ============================================================
# RUN OPTUNA — 50 trials per model
# Direction = maximize because we want the highest F1
# ============================================================
N_TRIALS = 50

print("Optimising Logistic Regression...")
study_lr = optuna.create_study(direction='maximize')
study_lr.optimize(objective_lr, n_trials=N_TRIALS)
print(f"  Best F1  : {study_lr.best_value:.4f}")
print(f"  Best params: {study_lr.best_params}")

print("\nOptimising Random Forest...")
study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=N_TRIALS)
print(f"  Best F1  : {study_rf.best_value:.4f}")
print(f"  Best params: {study_rf.best_params}")

print("\nOptimising XGBoost...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS)
print(f"  Best F1  : {study_xgb.best_value:.4f}")
print(f"  Best params: {study_xgb.best_params}")

print("\nOptimising Decision Tree...")
study_dt = optuna.create_study(direction='maximize')
study_dt.optimize(objective_dt, n_trials=N_TRIALS)
print(f"  Best F1  : {study_dt.best_value:.4f}")
print(f"  Best params: {study_dt.best_params}")

print("\nAll models optimised")

Optimising Logistic Regression...
  Best F1  : 0.4103
  Best params: {'C': 0.002427242465591971, 'solver': 'lbfgs'}

Optimising Random Forest...


[W 2026-04-08 08:26:06,676] Trial 7 failed with parameters: {'n_estimators': 500, 'max_depth': 14, 'min_samples_split': 7, 'min_samples_leaf': 1} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/opt/anaconda3/envs/anaconda_full/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/6t/jnkw6zz543q4vx7jvts4bb8h0000gn/T/ipykernel_68699/1834459007.py", line 28, in objective_rf
    score  = cross_val_score(model, X_train_sm, y_train_sm,
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/anaconda_full/lib/python3.12/site-packages/sklearn/utils/_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/anaconda_full/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 677, in cross_val_score
    c

KeyboardInterrupt: 

In [ ]:
# ============================================================
# TRAIN FINAL MODELS WITH BEST PARAMETERS FROM OPTUNA
# ============================================================

# Logistic Regression
lr_final = LogisticRegression(**study_lr.best_params,
                               class_weight='balanced',
                               max_iter=1000, random_state=42)
lr_final.fit(X_train_sm, y_train_sm)
lr_results = evaluate_model('Logistic Regression', lr_final, X_val_scaled, y_val, 'Blues')

# Random Forest
rf_final = RandomForestClassifier(**study_rf.best_params,
                                   class_weight='balanced',
                                   random_state=42, n_jobs=-1)
rf_final.fit(X_train_sm, y_train_sm)
rf_results = evaluate_model('Random Forest', rf_final, X_val_scaled, y_val, 'Greens')

# Feature importance — Random Forest
feat_imp = pd.Series(rf_final.feature_importances_, index=X_train.columns)
fig, ax = plt.subplots(figsize=(9, 5))
feat_imp.sort_values(ascending=False).head(10).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title("Top 10 Features — Random Forest")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# XGBoost
xgb_final = XGBClassifier(**study_xgb.best_params,
                            objective='multi:softprob',
                            num_class=3,
                            eval_metric='mlogloss',
                            random_state=42, verbosity=0)
xgb_final.fit(X_train_sm, y_train_sm)
xgb_results = evaluate_model('XGBoost', xgb_final, X_val_scaled, y_val, 'Oranges')

# Decision Tree
dt_final = DecisionTreeClassifier(**study_dt.best_params,
                                   class_weight='balanced',
                                   random_state=42)
dt_final.fit(X_train_sm, y_train_sm)
dt_results = evaluate_model('Decision Tree', dt_final, X_val_scaled, y_val, 'copper')

# Feature importance — Decision Tree
feat_imp_dt = pd.Series(dt_final.feature_importances_, index=X_train.columns)
fig, ax = plt.subplots(figsize=(9, 5))
feat_imp_dt.sort_values(ascending=False).head(10).plot(kind='barh', ax=ax, color='sienna')
ax.set_title("Top 10 Features — Decision Tree")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
all_results = {
    'Logistic Regression': lr_results,
    'Random Forest'      : rf_results,
    'XGBoost'            : xgb_results,
    'Decision Tree'      : dt_results,
}

results_df = pd.DataFrame(all_results).T

# Weighted scoring
weights = {'Recall': 0.35, 'F1 Score': 0.30,
           'Precision': 0.20, 'ROC AUC': 0.10, 'Accuracy': 0.05}

results_df['Weighted Score'] = results_df.apply(
    lambda row: round(sum(row[m] * w for m, w in weights.items()), 4), axis=1)

results_df = results_df.sort_values('Weighted Score', ascending=False)

print("=== Final Model Comparison ===")
print(results_df.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['gold' if i == 0 else 'steelblue' for i, _ in enumerate(results_df.index)]
bars = ax.bar(results_df.index, results_df['Weighted Score'], color=colors)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.002,
            f"{bar.get_height():.4f}",
            ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Weighted Score')
ax.set_title('Model Comparison — Weighted Score\n(Gold = Best Model)')
ax.set_ylim(0, max(results_df['Weighted Score']) * 1.2)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# All metrics comparison
metrics  = ['Precision', 'Recall', 'F1 Score', 'ROC AUC', 'Accuracy']
colors_m = ['steelblue', 'tomato', 'green', 'purple', 'orange']
x     = np.arange(len(results_df))
width = 0.15

fig, ax = plt.subplots(figsize=(12, 6))
for i, (metric, color) in enumerate(zip(metrics, colors_m)):
    ax.bar(x + (i-2)*width, results_df[metric], width,
           label=metric, color=color, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, rotation=15)
ax.set_ylabel('Score')
ax.set_title('All Metrics — All Models')
ax.legend()
ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.show()

In [ ]:
model_map = {
    'Logistic Regression': lr_final,
    'Random Forest'      : rf_final,
    'XGBoost'            : xgb_final,
    'Decision Tree'      : dt_final,
}

best_name  = results_df.index[0]
best_model = model_map[best_name]

print("=" * 60)
print("  BEST MODEL")
print("=" * 60)
print(f"  Model          : {best_name}")
print(results_df.loc[best_name].to_string())
print()
print("Why other models were not selected:")
for name in results_df.index[1:]:
    r = results_df.loc[name]
    print(f"  {name:<25} Weighted={r['Weighted Score']:.4f} | "
          f"Recall={r['Recall']:.4f} | F1={r['F1 Score']:.4f}")